# AVTR-1 conversational avatar server

Run the single code cell below. It prepares AVTR-1, downloads/builds the required models, and prints a temporary `gradio.live` URL. Open that URL to capture a portrait and voice reference, then use push-to-talk conversational turns.

Before starting: accept the model conditions at [avaturn-live/avtr-1](https://huggingface.co/avaturn-live/avtr-1), select an **A100 or L4 GPU** runtime, and add `HF_TOKEN` to Colab Secrets. Optionally add `GRADIO_PASSWORD`; otherwise the cell generates and prints a random password. Only clone a face and voice with the person's explicit permission.

In [ ]:
# One-time setup followed by the persistent Gradio server. Keep this cell running.
import base64, getpass, json, os, secrets, shutil, subprocess, sys, time, zlib
from pathlib import Path

try:
    from google.colab import userdata as colab_userdata
except ImportError:
    colab_userdata = None

def read_secret(name, *, report_error=False):
    if value := os.environ.get(name):
        return value
    if colab_userdata is None:
        return None
    try:
        return colab_userdata.get(name)
    except Exception as exc:
        if report_error:
            print(f"Colab secret {name!r} is unavailable ({type(exc).__name__}). Check its name and enable notebook access in the Secrets panel.", flush=True)
        return None

SETUP_STAGES = 7
setup_started = time.monotonic()
def stage(number, title, detail):
    elapsed = time.monotonic() - setup_started
    print(f"\n{'=' * 72}\n[{number}/{SETUP_STAGES}] {title}  (elapsed {elapsed / 60:.1f} min)\n{detail}\n{'=' * 72}", flush=True)

stage(1, "Checking the GPU", "This should finish almost immediately.")
if shutil.which("nvidia-smi") is None:
    raise RuntimeError("Choose Runtime > Change runtime type > GPU, then reconnect.")
subprocess.run(["nvidia-smi"], check=True)

REPO = Path("/content/avtr-1")
PIXI = Path.home() / ".pixi/bin/pixi"
STORAGE = Path("/content/avtr1_storage")
AGENT_VENV = Path("/content/avtr_agent_venv")
AGENT_PYTHON = AGENT_VENV / "bin/python"
configured_repo_url = read_secret("AVTR_REPO_URL")
REPO_URL = configured_repo_url or "https://github.com/avaturn-live/avtr-1.git"
LLM_MODEL = "Qwen/Qwen3-1.7B"
STT_MODEL = "small"
TTS_MODEL_REPO = "ResembleAI/chatterbox"  # Multilingual V3; shared by English and Dutch.
STT_MODEL_REPO = "Systran/faster-whisper-small"  # Multilingual model; about the same size as small.en.
TTS_ALLOW_PATTERNS = ["ve.pt", "t3_mtl23ls_v3.safetensors", "s3gen.pt", "grapheme_mtl_merged_expanded_v1.json", "conds.pt", "Cangjie5_TC.json"]

stage(2, "Preparing the AVTR-1 checkout", "Cloning on the first run; otherwise pulling updates.")
if REPO.exists():
    subprocess.run(["git", "-C", str(REPO), "remote", "set-url", "origin", REPO_URL], check=True)
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)
if not PIXI.exists():
    subprocess.run(["bash", "-lc", "curl -fsSL https://pixi.sh/install.sh | bash"], check=True)
stage(3, "Installing the locked AVTR environment", "Pixi will reuse packages already cached in this runtime.")
subprocess.run([str(PIXI), "install"], cwd=REPO, check=True)

hf_token = read_secret("HF_TOKEN", report_error=True)
gradio_password = read_secret("GRADIO_PASSWORD")
gradio_username = read_secret("GRADIO_USERNAME") or "avatar"
if not hf_token:
    hf_token = getpass.getpass("Hugging Face read token: ")
if not hf_token:
    raise RuntimeError("HF_TOKEN is required for the gated AVTR-1 weights.")
if not gradio_password:
    gradio_password = secrets.token_urlsafe(12)

env = os.environ.copy()
env.update({
    "HF_TOKEN": hf_token,
    "HF_HUB_TOKEN": hf_token,
    "AVTR1_LOCAL_STORAGE": str(STORAGE),
    "GRADIO_USERNAME": gradio_username,
    "GRADIO_PASSWORD": gradio_password,
    "PYTHONUNBUFFERED": "1",
})
stage(4, "Downloading AVTR artifacts", "Large first-run download; existing files are checked and reused. Hugging Face prints per-file progress below.")
subprocess.run(
    [str(PIXI), "run", "python", "scripts/download_artifacts.py"],
    cwd=REPO, env=env, check=True,
)
engine_root = STORAGE / "main"
speech_engine_dir = engine_root / "speech2motion_runtime_artifacts_cc"
renderer_engine_dir = engine_root / "renderer_runtime_artifacts_cc"
avtr_engines = [speech_engine_dir / "avtr1_encode_fp16.engine", speech_engine_dir / "avtr1_decode_fp16.engine"]
renderer_engines = [renderer_engine_dir / name for name in ["decoder_b5_fp16.engine", "warp_network_b5_fp16.engine", "modnet_b5_fp16.engine", "stitch_network_b5_fp16.engine"]]
hubert_engine = speech_engine_dir / "hubert_lbs_fp16.engine"
stage(5, "Preparing TensorRT engines", "AVTR motion, renderer, and HuBERT engines are required. TensorRT may appear quiet while optimizing a graph; keep the cell running.")
if not all(path.is_file() for path in avtr_engines):
    print("Building the two AVTR motion engines.", flush=True)
    subprocess.run([str(PIXI), "run", "build-trt-engines-avtr1"], cwd=REPO, env=env, check=True)
else:
    print("Cache hit: reusing both AVTR motion engines.", flush=True)
if not all(path.is_file() for path in renderer_engines):
    print("Building the four renderer engines, including the required GridSample3D warp engine.", flush=True)
    subprocess.run([str(PIXI), "run", "build-trt-engines-renderer"], cwd=REPO, env=env, check=True)
else:
    print("Cache hit: reusing all four renderer engines.", flush=True)
if not hubert_engine.is_file():
    print("Building the required HuBERT TensorRT engine.", flush=True)
    subprocess.run([str(PIXI), "run", "build-trt-engines-hubert"], cwd=REPO, env=env, check=True)
else:
    print("Cache hit: reusing the HuBERT TensorRT engine.", flush=True)

stage(6, "Installing the conversational stack", f"Creating an isolated environment, fixing Perth's runtime dependency, and downloading {LLM_MODEL}, {TTS_MODEL_REPO}, and {STT_MODEL_REPO}. Existing caches are reused.")
venv_ready = AGENT_PYTHON.exists() and subprocess.run(
    [str(AGENT_PYTHON), "-m", "pip", "--version"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
).returncode == 0
if not venv_ready:
    if AGENT_VENV.exists():
        print("Removing the incomplete assistant environment left by a previous attempt.", flush=True)
        shutil.rmtree(AGENT_VENV)
    print("Creating the assistant environment with AVTR's managed Python 3.12.", flush=True)
    subprocess.run(
        [str(PIXI), "run", "python", "-m", "venv", str(AGENT_VENV)],
        cwd=REPO, env=env, check=True,
    )
subprocess.run([str(AGENT_PYTHON), "-m", "pip", "install", "--upgrade", "pip", "setuptools==80.9.0", "wheel"], check=True)
subprocess.run([
    str(AGENT_PYTHON), "-m", "pip", "install",
    "chatterbox-tts==0.1.7", "faster-whisper>=1.1,<2", "accelerate>=1,<2",
], check=True)
subprocess.run([
    str(AGENT_PYTHON), "-c",
    "import perth; assert perth.PerthImplicitWatermarker is not None, 'Perth watermarker import failed'",
], env=env, check=True)
for model_repo in [LLM_MODEL, TTS_MODEL_REPO, STT_MODEL_REPO]:
    print(f"Downloading/checking model cache: {model_repo}", flush=True)
    subprocess.run([
        str(AGENT_PYTHON), "-c",
        "import json,sys; from huggingface_hub import snapshot_download; snapshot_download(repo_id=sys.argv[1], allow_patterns=json.loads(sys.argv[2]) or None)",
        model_repo, json.dumps(TTS_ALLOW_PATTERNS if model_repo == TTS_MODEL_REPO else []),
    ], env=env, check=True)

# Portable copy of scripts/gradio_colab_app.py, embedded so this notebook
# works even while the upstream checkout does not yet contain our new file.
APP_ZLIB_B64 = "eNq9O2tv4za23/MreNUPI83KijPTbmcD+ALZ6XRQYB7ZxN2iyAYCLdO2GlnSUlQcb26A+x/2H+4v2XP4kEiJdma6F3eKxhIfh4fnxfOgvvmv07bhp4u8PGXlPan3YlOVr0+CIHjP6TKvyIpXpSCsXJJVxQklouXlZLGf4G9M7qs8Y5OsqEq2JBd/nV9Nzgi9p4Ly5ORkvskbUvMqY01DeFs2JC+J2DBS5OuN2DH8S7KqvGe8oSKvSlpMttWSFbDcfQ7rblkpEg32hAMOMBBh5KIh1a4kl/lDbo8lTSX75qxsKn41P73czyuebUidw+KNoHuSN1VBBVsmuMWTE9jdlqTpqoXtsDQl+bauuCC0LCshUWpOTkwbX9eUN8y8rzPz9FtTlea5asxTs2lFXnRv7UJTwrSIDWdA4XJtGto2XyqEaio2Rb4w2FzCq+oQe9jJ2rRflPsOu7XiFm3gqVsB9+680FYPElQBvPzpg4H205auWax+Ptew75PrX6/n7z6ml1efP17OyYyEJwT+Bb9WLRCDgSyseA5MKYCqpWCcZiK/Z4b9gF2zYxzZBcJBQHbErgKiwFIxKSnQmxYkUBCburpjJWmAhawEGiXk54aRuqAoL+xBkF0OYtkKsqX8bgmcj0GEGtHEyNM1I8ucs0xyKzYgYT22rX7Lk+Akgq1kBQUZvJCoXTMOEncuxy3ZCtifl7lIU7U9/NewYhV3by/7R87q6lzyo2+rQQiHbQ1QGxAbNu8qfpcCssP2otimUvDPYSa3wYhRe0Qm/00+AUnPHXQTRA14hD/wDFJ+z8LIHYKYwhD8OTREIw6j9NOhgWYrMNI8Hhra7S4t6RZBdw3DtYU7rmvwL51s7+BvCEoJctPM5rwF8WUPIBhpdSdfB4ik67pNiyq7A9id9iUfoGGIcgpLwyAk86BDoKTm/2Dc3w1bOzAPzJLu6AWPsxXjKPHpisOWG9xViOMlk1FGeiZvqcg2DIGAaQMDFtr8StZFtQiDl6dK+ZqUcpGvQCGb0+EaQdTvNV8RsHMGdL+WFHWagw6uefKO84qHTp/UMDTLBC0HjETrqBeUpmGbNw3Slly1yuLDMmxRVXdADdHWp6jDRUWXJGNFQVY5b0QSOEtEls7hUWOQvJne9gRkJa+K4pDeGtxSNKdSg8j/SA70Q+TxdaBf8gCQLdjNMs/EDWogmtzbWCqjPd56vj0fUtdF4xiNgzm9k5YSFkXi0J68mkQj1lkbOAr5imUVXzqws4LRUh/gnZBAa173q1mUBYZWZZovQQDxoErwz7dhlGzYw2iUsguunTi1YPgmHNfmH2nRWOqsxFxhswoyONIX6WMP/ykYCwEeHxonv95FgOIqeOxAPyV1uQ4GomKg9Ps8JYHq6oAmO3of9KQTfO/yBg8zdcwmVc3K0BGQCE/nHDvPRyonm2F5c0QnQJ5VCnNLOEAbFsr+KFEelQCmv/+zJTEOlKShYKht4sTo34GSzYLLT+8H0xTVNNxUOhEhWqewF78o7ikE+kC3oDcpB09r9urbdDqd9gDZQ8ZqQd7JH6Ciu0/lNSV8KzhjoUXomOTrsgInjaFENwP7PmR10pZFXt6F2hCND4ROSSwZB48PyfvoDAp6sQrOLRmO/aMA00DaEhv3aDC4kzEY2j0PxpjdaGj25obgBvKnZ3TsGA4HrxxOjT0Mu7kddKGlhfZp3/w0tMPuSSBpNgByBQfrHky/MjmU/L1lDTLaeP9vLY8fCCodtySIB3CPbvnQ/qL+bABnhAl5SMYKy3PimnFtsI+YefQxe2Mfa1s/tvIK/JguIEsIMpQv0bj/pmPFLQy9ufWNkBzB7umQEWoJsOiPTzFykgSB+r9H03I1XOXVZKlanhknFYYBl0rJFdP00tHkc3TzPe7narWt2RrNotLd3SbPNmGgmsdHlmr3HVfgLIh8y/SRpcFCCImzIIATtCggyuykCIw+RpU4JbDdhaYtpJHuIq4ERrliezOyimq1eOzmTPaBrzX3tEqtlzSNfFNo5oN05gVP+SH4PT8i/wiLj4MRA23PaC2jXlChuhXqxHUGYOg1aHa4qUidKHnMwFF/lqmr4G3VFkvJ0SXDKZKXUijPyaMG2AgI9PnN5OzVdHp++xRYWp2i86KCgkZLMTB5DRObc3R3LvWLR0xRUTvfHqTK7ZUHiJ4cTpPpd1IhslnwoZJhAmlqxrIN7Dmr1hAuAnUHh6SMp1e0gUg4BRWA8dyE1r+o148ylvGcrCrasEeN/W1PiIQo3oMRnAVZu6Sg+Vm1BU6yVOxraFwBrcTZHw/51R09+pBGUwWNSh/OPEeqs1dDUrkpHSIR9hFLei7odci0js5ptKKSFPix4m9p29Diw8dYts4Nmj4K2mGZMzjBldIaZBTEsTRhkxuS+pwdFcj5sBlB9LPKXWKsqjIfky4Vqyis41F4xd50S2vvEB8vIQB7jmGvpkOG9Vk8fDvIr2xDBUj3onpIYJ0UtH5RGb697frm2DyfX3vZJKPg8dgRTR3BtvVfykzG8wXT6i9tRx/CScWHXzs7ssbcYBOTtAsAQI8SC5KDaQ8wJgtGt2kDcjQDe3APlmeVF4C5NIonB0PVgATJb1VehnrpBA0p6C7Pa4gyVlK7ZAeeZQa9yAxw8gPg2JdLvVPjR6lkENG+w7l0UyzfBf7c3o7JAGFOA44/0t89/R4DCKLRaQyaPRivLXoRoMECkIJGJwf45MrfS43CzeTN+eBs6YG2DeMuSLMPC1rv+oCcbus+Uuv0OqF1XexTFMEUkKwxievyzWxwcIZpACqCdPvocpmuWcm4tFSpWtpzDrKSLuDMFRuIKEBBhqAsNSzB/DZj5MMbBRucNCUisAXMUDezoBZBlAjlkUmrkyjR74HKcFFaiyQvTdyKOhpGrorrrbBlhwCCM61jM/XypcJ3bHe29CEt2U5toJn96Y8e01SlyhXx0Ev5DuBPcUzzstk0+d5n/+q0hq43/q672avpuIezmolccQu4Uoj97CyZvokPpo5QZvIlsqSjzs309kZtPJE/2J80G1oz8DhuyfmtO90jispzCTvooHN3eZ3C4Z3ltDBUk/Fmp9S9SS+0mMQ9SnGP6dCWyA7LJDT7EjwmNEnaKsgB2iR0IaB+V56d8uY9HtEXS9aO3vdkALp1MiUX72ywlHFpOWcdJv3OBVVJB/RRFWJRjJCTrG5DeOzBN9ylFwxyjCLWgb4gWd9F1sOsuvTkhjl4m1Z9a7Zap7hAql1U6VP13WUFLi6cSW2Zpf9wen3k5mDDpIeywtqTPjjVT5EvElkuOlj/QemCqe7wBFtTeXjCycQwuS1tWfiC3gt+Zqwb+PirVQEH64tYAgtf4MFXi+Z0OCCp9y+i6ASEoC3YeDnVrhbEpUP8E0ncEvTMQTnYA2CkxoXqJzoBvlFw7DXr0krmfVAp1YDkSrZ/Vs04GgJy8AIG48OXL+92lK8bLZ3qJWnrJYqiy6nZo+TFoDWCSNlhmRnmNEZPSvy0BvqR75E58W0CKwyeLZixW/BywCigGPSHDOpY1SS6nplkVb23DAc038ic+1n64fPbiw/p9fzz1cX7dzI5oPJNfT3gPw6HO4BYLPJFswDDF7iq+rE3pPVFv1olfMMnSn09kw5kzPQ0pd2HomfZ6Y3OJ4u1d0NYhMR4TjDvriZgNw4sZozc0Rh8t5x15buh13E/g///f2P2JRM0L1B8QycYx5Cwb8Ck283k9RTD8+dDflkoUnqA4cUKFmDL87+Vj3oxJ8BHtA7ZduM4GmPsq+gcS/X9H5h1O5RyEg7oYfRvYTSqIfVpxP6Pz3OPB7iPC0pd4q97gQNZhKM0cHS8JPROFs/sIhMtlyoUhLgHohSGwLlAnlEnpvfUoQas+ZJalJUYHtebvpG3Pk7NRQ6JGmaKfKmFhlDYB3hwgDFmL95f/py4Po5b/XWxG5VnrByAlWsyTB/XU+zI+o2JrOcmuETq7auWd5v1VGQ6OnQ+Vh+auoT1VHNc+o+34uPAp8qks3a0QZ1nGV6JIZcFozCQKwYhvTvU6BrsoMMiz/6/7VJBv/BcC46KYRvm2bcOH/FGADyFliibvDgmtqNIxpe+WMDyzk2obBDu4uODJFOO8xF6uenoOd5bouW6xRqcuqWkjCjEW7QkEO6IvYLpI5IM+sp2u5D+X146u5U5/phMo4j8gZx5hNGuqkr/TRcH7KLTbeSnkBKcUdFyFajeRwuz8+nr5ZOqXvpB3edL9jWgtvW3wVGB+a5TmLc6+YRCJ22QR1507qaPgHT4oakxtH+3sU2BMbh1Bj5WUYDsh5E/PZdg+imRrE0zmm1YeFz+v+/k/6o77mR+W3osRJLv4L6GcY3X5ZnpvfYVxNvYO0Pp98wigH+cdiQs/vrHDdxr99U/xfW0nbfxBA9htQJDRCGANH7C3Hhbf0cG6vB0CrIOVIeJDgxJsQMAbj3bG3uIxyuEB6qEdsrt0PiuZmgp5GGxPUu6fPAPcHwGPjviqfwODy8/JZSG+iUUXGRbP58bJYXzwChNlUMgRjXqMUv0rQRzRo7PBXVZwDOlu8iANzeg7cDU/vjFLcE4sPYy2IfHkxMZ9C7avFimbQ6Rl7yh6NxXlK4kwPgzujD6rpZ0brq2UOSiYJjVz8A10pdxBw6TDqrkLZMl21qempFHAHeNj+HjU08haPyor166UhB845Tz+3XVQn8r3aMkeAdD95hIlYVdmXXipqYLDU5RNyE/ol+ob3kNHdXAcwdCV6tjNLolafEOadtsJqKaCFrcSXVoEvK5BL8BOynEIjCpd3slPQeA0X5DiNpU5Qtkb13kWS5vLeNtVevSmmXADFvmdBEGZwnRnvbYzx666GbiVbUbpuDsCy6KS/IWkF8ngYismAWX5grJAb2QpepmdhPs2CKjMvGvbocdOlR0XRHiN8xQBfEXmLnBoaxQv5CXEI6h/lZeTnvzr//959l3eEek6nikWPzslrZ5xqt6gwbtP9+WrIDp21HoHj1rTCTD00UrhPTqUUXlcxhc03vmkwQsL/EcTplZUPN8S/k+8MHEUlGKqtrqiHMOp+Oiegg13a5lF0CzbmQPL89JQgEay9QnUAYQDugQ/VKAikUWjx1oqrd30LxA/Vr0KnHszFBx3CjpqJBpjEZBqC+J89XS9KwkHZYil5y0ufOKz0VzZ2z4sxJjCHiR4VYxKxx8rORBJVr0tQEC3j/UhB+bG9fH00dDkR90Us+S72LyOpGV0qJls1fJ9+ibM6zynMWG8Ncq8Fy3+ZKiEHyBDXFcx+fxwJoSIPO9wUPiNcJDkwIi7fWzSDxjl024XguvTv5atc/rj87VylDZC+XCcP0ZQM/gatYYa0pneeW3PF3Y/lWYdxEiQP0rPoadumFwg6HyF24DC70KDt4WWFSiw882A7HWOFP7HWiAvAPo1SM8X7jjHBkf0o7eHTueZHD03w2uP0rXLFHjXH1WRb7ZjTGh8fAoHJgPFYjBBOWuju19PDDasW1zB8BonctrJ4dL1b2BObIv+Y2Xd1euwdUheDwwGYNiy6EN9+oTd4IUDwQ1HohYLAVErzvMqVdl1nKk8T4t8m0uZmdfSx1bcI7QRw7zE8iH2JDFagu/Z/tfxe6ujrWtdKAhv2NLsXoVyrDCfNqWfMLb8DU4xcpwyEZMOnUDLvi6RZm8lD1411BiDpIwS9NllaVpZM1M8IYF1VPCYDLBKodRWnXFlLO/tzkYBut69oG5WIn6vXN1Wex3TT+xij34NcMEs20OIF0hlG9hcKpzBKdYgk3Vd3mpzpgZA3UUWdB69Rlk0IMO/rJj5Sn+eT2Bo83c6z+4XTGG0GxpUYCpOj61rCYykw4zqbyZDRMFXrsHJTa5AS1QGoItTFq+VHnTrYDjAPz4zB5tZWOtfFvepHBK5AVeurGPMH9WFqJGWteGmfCig8j3lz8PbgcrpcVbfVZYHTrf9s1kRdmtxaHcqXZ8Gn3jp7r0y/hTP9Vt3jyf/KkB3avn4z+zgrBHRLpitsXTdpA4UH2Y9tIf0llVZZlwfn918cNPn9Ofr99dfbr4+A59WpMY0LLRNDt0jw9Ovby4vv7l89UPegJtwemYkdCsGXcgImRwB4+Beeq/jkPsEzhJWhbijSN5z+5NJ7Gpz45LU2d5rUlB4WzZ2BVDpICyhsE0kf9ZnraU7RkKnCRqCZqJLdalEdjJDP+4Tan2MCCo5vk9ZkqUlOnMbs0rUaFFsJYCbat2eF6DVZA2n4eOLPSfS0a3NoLVTn3iYlVxUa9y/FAVt5WmZDYjQZqilqVpoBRE3yj4N3uYEf8="
app_path = REPO / "scripts/gradio_colab_app.py"
app_source = zlib.decompress(base64.b64decode(APP_ZLIB_B64)).decode()
app_source = app_source.replace(
    'portrait_out = self._reference_frames_dir() / f"{avatar_id}.png"\n        voice_out',
    'portrait_out = session_dir / "portrait.png"\n        registered_portrait = self._reference_frames_dir() / f"{avatar_id}.png"\n        voice_out',
).replace(
    'image.save(portrait_out, format="PNG")\n            self._convert_audio',
    'image.save(portrait_out, format="PNG")\n            shutil.copy2(portrait_out, registered_portrait)\n            self._convert_audio',
).replace(
    'portrait_out.unlink(missing_ok=True)',
    'registered_portrait.unlink(missing_ok=True)',
).replace(
    '"portrait": str(portrait_out),\n            "voice_reference"',
    '"portrait": str(portrait_out),\n            "registered_portrait": str(registered_portrait),\n            "voice_reference"',
)
app_source = app_source.replace('api_name=False', 'api_visibility="private"')
app_source = app_source.replace('import subprocess\nimport threading', 'import subprocess\nimport sys\nimport threading\nimport time')
app_source = app_source.replace(')\n\n\nclass AvatarServer:', ')\nTTS_REPO_ID = "ResembleAI/chatterbox-turbo"\nTTS_ALLOW_PATTERNS = ["ve.safetensors", "t3_turbo_v1.safetensors", "s3gen_meanflow.safetensors", "conds.pt", "*.json", "*.txt", "*.model"]\n\n\nclass AvatarServer:')
app_source = app_source.replace('if self._stt is None:\n            progress', 'if self._stt is None:\n            started = time.perf_counter()\n            progress')
app_source = app_source.replace('compute_type="float16"\n            )\n        if self._tokenizer', 'compute_type="float16"\n            )\n            print(f"Loaded {self.stt_model_name} in {time.perf_counter() - started:.1f}s", flush=True)\n        if self._tokenizer')
app_source = app_source.replace('if self._tokenizer is None or self._llm is None:\n            progress', 'if self._tokenizer is None or self._llm is None:\n            started = time.perf_counter()\n            progress')
app_source = app_source.replace('device_map="auto",\n            )\n        if self._tts', 'device_map="auto",\n            )\n            print(f"Loaded {self.llm_model_name} in {time.perf_counter() - started:.1f}s", flush=True)\n        if self._tts')
app_source = app_source.replace('if self._tts is None:\n            progress', 'if self._tts is None:\n            started = time.perf_counter()\n            progress')
app_source = app_source.replace('from chatterbox.tts_turbo import ChatterboxTurboTTS\n\n            self._tts = ChatterboxTurboTTS.from_pretrained(device="cuda")', 'from chatterbox.tts_turbo import ChatterboxTurboTTS\n            from huggingface_hub import snapshot_download\n\n            local_path = snapshot_download(repo_id=TTS_REPO_ID, allow_patterns=TTS_ALLOW_PATTERNS, token=os.getenv("HF_TOKEN") or None)\n            self._tts = ChatterboxTurboTTS.from_local(local_path, device="cuda")\n            print(f"Loaded Chatterbox Turbo in {time.perf_counter() - started:.1f}s", flush=True)')
app_source = app_source.replace('chat = gr.Chatbot(label="Conversation", type="messages")', 'chat = gr.Chatbot(label="Conversation")')
app_source = app_source.replace(
    '    args = parse_args()\n    if not torch.cuda.is_available():',
    '    args = parse_args()\n    print(f"Server preflight: Python {sys.version.split()[0]} | Gradio {gr.__version__} | PyTorch {torch.__version__}", flush=True)\n    print(f"Repository: {args.repo}\\nStorage: {args.storage}", flush=True)\n    if not torch.cuda.is_available():',
).replace(
    '        raise RuntimeError("This app requires a Colab GPU runtime")\n    server = AvatarServer(',
    '        raise RuntimeError("This app requires a Colab GPU runtime")\n    print(f"CUDA ready: {torch.cuda.get_device_name(0)}", flush=True)\n    server = AvatarServer(',
).replace(
    '    demo = build_ui(server)\n    username =',
    '    demo = build_ui(server)\n    print("Gradio interface constructed successfully; opening share tunnel...", flush=True)\n    username =',
)
if app_path.exists():
    print(f"Using the readable server script checked into the cloned repository: {app_path}", flush=True)
else:
    print("Server script is absent from the cloned upstream repository; using the notebook fallback copy.", flush=True)
    app_path.write_text(app_source)

stage(7, "Launching the Gradio server", "Setup is complete. Keep this cell running for as long as you use the interface.")
print(f"Total setup time: {(time.monotonic() - setup_started) / 60:.1f} minutes", flush=True)
print("Open the gradio.live URL that appears below.", flush=True)
print(f"Login: {gradio_username} / {gradio_password}\n", flush=True)
server_process = subprocess.Popen([
    str(AGENT_PYTHON), str(app_path),
    "--repo", str(REPO),
    "--pixi", str(PIXI),
    "--storage", str(STORAGE),
    "--llm-model", LLM_MODEL,
    "--stt-model", STT_MODEL,
], env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
assert server_process.stdout is not None
for line in server_process.stdout:
    print(line, end="", flush=True)
server_returncode = server_process.wait()
if server_returncode:
    raise RuntimeError(f"Gradio server exited with status {server_returncode}; see its full exception immediately above.")
